# PySpark Job: clean raw orders, join products, write partition parquet (silver) and daily sales (gold)

In [1]:
import sys
from pyspark.sql import SparkSession, Window, functions as F

In [2]:
import os
import glob

os.environ["HADOOP_HOME"] = "D:\\hadoop"
os.environ["PATH"] += os.pathsep + "D:\\hadoop\\bin"

In [3]:
day = "2026-11-01"

In [4]:
spark = (SparkSession.builder.appName("shopflow-transform")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.source.partitionOverwriteMode", "dynamic")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

In [5]:
path =  "../data/raw/orders" if day == "all" else f"../data/raw/orders/dt={day}"

In [ ]:
print(path)

In [6]:
raw = spark.read.json(path)
raw.printSchema()
print(f"day {day}: raw {raw.count()}")

root
 |-- customer_id: long (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- qty: long (nullable = true)
 |    |    |-- sku: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- ordered_at: string (nullable = true)
 |-- status: string (nullable = true)

day 2026-11-01: raw 6708


In [7]:
orders = raw.dropDuplicates(["order_id"])

In [ ]:
# try:
#     orders.show(10, truncate=False)
# except Exception as e:
#     print(str(e)[:2000])

In [8]:
lines = (orders
         .withColumn("item", F.explode("items"))
         .select("order_id", "customer_id", "status",
                 F.to_timestamp("ordered_at").alias("ordered_at"),
                 F.col("item.sku").alias("sku"),
                 F.col("item.qty").alias("qty")
                 )
         )
print(f"day {day}: raw {raw.count()}, orders {orders.count()}, lines {lines.count()}")

day 2026-11-01: raw 6708, orders 6695, lines 6695


In [10]:
lines.show(10, truncate=False)

+----------------+-----------+---------+-------------------+-------+---+
|order_id        |customer_id|status   |ordered_at         |sku    |qty|
+----------------+-----------+---------+-------------------+-------+---+
|2026-11-01-00001|986        |paid     |2026-11-01 21:36:00|SKU-019|3  |
|2026-11-01-00002|1119       |paid     |2026-11-01 03:57:00|SKU-034|2  |
|2026-11-01-00003|239        |paid     |2026-11-01 16:32:00|SKU-017|1  |
|2026-11-01-00004|1658       |refunded |2026-11-01 03:21:00|SKU-024|0  |
|2026-11-01-00005|1366       |paid     |2026-11-01 04:36:00|SKU-010|2  |
|2026-11-01-00006|1532       |refunded |2026-11-01 14:27:00|SKU-018|1  |
|2026-11-01-00007|723        |paid     |2026-11-01 14:11:00|SKU-035|2  |
|2026-11-01-00008|1857       |cancelled|2026-11-01 06:03:00|SKU-019|1  |
|2026-11-01-00009|208        |paid     |2026-11-01 13:46:00|SKU-012|1  |
|2026-11-01-00010|611        |paid     |2026-11-01 04:54:00|SKU-020|2  |
+----------------+-----------+---------+-----------

In [14]:
products = (spark.read.option("multiLine", True)
            .json("../data/raw/products.json"))
joined = (lines.join(F.broadcast(products.select("sku", "category", "price")), "sku", "left")
          .withColumn("order_date", F.to_date("ordered_at"))
          )
is_bad = (F.col("qty") < 0) | F.col("price").isNull() | F.col("status").isNull()
bad = joined.filter(is_bad)
silver = (joined.filter(~is_bad)
          .withColumn("amount", F.round(F.col("qty") * F.col("price"), 2))
          )
silver.show(5, truncate=False)
print(f"day {day}: raw: {raw.count()}, orders {orders.count()}, lines {lines.count()}")


+-------+----------------+-----------+--------+-------------------+---+--------+------+----------+-------+
|sku    |order_id        |customer_id|status  |ordered_at         |qty|category|price |order_date|amount |
+-------+----------------+-----------+--------+-------------------+---+--------+------+----------+-------+
|SKU-019|2026-11-01-00001|986        |paid    |2026-11-01 21:36:00|3  |Gaming  |85.1  |2026-11-01|255.3  |
|SKU-034|2026-11-01-00002|1119       |paid    |2026-11-01 03:57:00|2  |Gaming  |557.26|2026-11-01|1114.52|
|SKU-017|2026-11-01-00003|239        |paid    |2026-11-01 16:32:00|1  |Audio   |170.17|2026-11-01|170.17 |
|SKU-024|2026-11-01-00004|1658       |refunded|2026-11-01 03:21:00|0  |Gaming  |824.53|2026-11-01|0.0    |
|SKU-010|2026-11-01-00005|1366       |paid    |2026-11-01 04:36:00|2  |Laptops |608.48|2026-11-01|1216.96|
+-------+----------------+-----------+--------+-------------------+---+--------+------+----------+-------+
only showing top 5 rows
day 2026-11-0

In [15]:
paid = silver.filter(F.col("status") == "paid")
daily = (paid.groupBy("order_date", "category")
         .agg(F.count_distinct("order_id").alias("orders"), 
              F.sum(F.col("qty").alias("units")),
              F.round(F.sum("amount"), 2).alias("revenue")
              )
         )
daily.orderBy("order_date", "category").show()

w = Window.partitionBy("order_date").orderBy(F.desc("revenue"))
top = (paid.groupBy("order_date", "sku").agg(F.round(F.sum("amount"), 2).alias("revenue"))
       .withColumn("rank", F.row_number().over(w)).filter("rank <= 3")
       )

top.orderBy("order_date", "rank").show()


+----------+-----------+------+-----------------+---------+
|order_date|   category|orders|sum(qty AS units)|  revenue|
+----------+-----------+------+-----------------+---------+
|2026-11-01|Accessories|  1069|             1470|618701.37|
|2026-11-01|      Audio|  1065|             1485|551609.95|
|2026-11-01|     Gaming|  1027|             1458| 849779.3|
|2026-11-01|    Laptops|  1037|             1434|669018.58|
|2026-11-01|     Phones|  1070|             1471|613805.71|
+----------+-----------+------+-----------------+---------+

+----------+-------+---------+----+
|order_date|    sku|  revenue|rank|
+----------+-------+---------+----+
|2026-11-01|SKU-004| 164898.3|   1|
|2026-11-01|SKU-009|162836.24|   2|
|2026-11-01|SKU-038|159847.35|   3|
+----------+-------+---------+----+



# Write into parquet

In [17]:
silver.write.mode("overwrite").partitionBy("order_date").parquet("../data/lake/silver/order_lines")
bad.write.mode("overwrite").partitionBy("order_date").parquet("../data/lake/quarantine/order_lines")
paid = spark.read.parquet("../data/lake/silver/order_lines").filter(F.col("status") == "paid")
daily.write.mode("overwrite").parquet("../data/lake/gold/daily_category_sales")

In [ ]:
spark.stop()